In [1]:
import ROOT
import numpy as np

In [2]:
file_data = ROOT.TFile("Pileup_data/MyDataPileupHistogram.root", "READ")

In [3]:
chain = ROOT.TChain("Events")
chain.Add("/eos/user/t/tdeandra/skim_outputs/Merged_Eras/Merged_mc_signal_2022.root")

1

In [4]:
c1 = ROOT.TCanvas()
hist_data = file_data.Get("pileup")
hist_data.Draw()
c1.Draw()

In [5]:
c2 = ROOT.TCanvas()
hist_mc = ROOT.TH1F("h_mc_pileup", "MC Pileup Distribution;Pileup_nTrueInt;Events", 100, 0, 100)
chain.Draw("Pileup_nTrueInt >> h_mc_pileup")
c2.Draw()

In [6]:
if hist_data.GetNbinsX() != hist_mc.GetNbinsX():
    if hist_data.GetNbinsX() > hist_mc.GetNbinsX():
        hist_data.Rebin(hist_data.GetNbinsX() // hist_mc.GetNbinsX())
    else:
        hist_mc.Rebin(hist_mc.GetNbinsX() // hist_data.GetNbinsX())

print(f'Número de bins no histograma de dados: {hist_data.GetNbinsX()}')
print(f'Número de bins no histograma de MC: {hist_mc.GetNbinsX()}')

hist_data.Scale(1.0 / hist_data.Integral())
hist_mc.Scale(1.0 / hist_mc.Integral())

print(f'Área no histograma de dados: {hist_data.Integral()}')
print(f'Área no histograma de MC: {hist_mc.Integral()}')

Número de bins no histograma de dados: 100
Número de bins no histograma de MC: 100
Área no histograma de dados: 1.0000000000000002
Área no histograma de MC: 0.9999999999389502


In [7]:
c1 = ROOT.TCanvas("c1", "Pileup Comparison", 800, 800)
pad1 = ROOT.TPad("pad1", "pad1", 0, 0.3, 1, 1.0)
pad1.SetBottomMargin(0.02)
pad1.SetLeftMargin(0.15)
pad1.Draw()
pad1.cd()

hist_mc.SetLineColor(ROOT.kRed)
hist_mc.SetLineWidth(2)
hist_mc.SetFillColorAlpha(ROOT.kRed, 0.35)
hist_mc.SetTitle("")
hist_mc.SetStats(0)

hist_data.SetMarkerStyle(ROOT.kFullCircle)
hist_data.SetMarkerSize(1.0)
hist_data.SetLineColor(ROOT.kBlack)

hist_mc.GetYaxis().SetTitle("Normalized Events")
hist_mc.GetYaxis().SetTitleSize(0.05)
hist_mc.GetYaxis().SetTitleOffset(1.3)
hist_mc.GetYaxis().SetLabelSize(0.04)
hist_mc.GetXaxis().SetLabelSize(0)
hist_mc.SetStats(0)
hist_data.SetStats(0)

hist_mc.GetYaxis().SetRangeUser(0, 0.08)
hist_mc.Draw("HIST")
hist_data.Draw("E1 SAME")

legend = ROOT.TLegend(0.65, 0.70, 0.88, 0.88)
legend.AddEntry(hist_data, "Data", "lep")
legend.AddEntry(hist_mc, "MC Simulation", "f")
legend.SetBorderSize(0)
legend.SetTextFont(42)
legend.SetTextSize(0.04)
legend.Draw()

cms_text = ROOT.TLatex()
cms_text.SetNDC()
cms_text.SetTextFont(61)
cms_text.SetTextSize(0.05)
cms_text.DrawLatex(0.15, 0.91, "CMS")

extra_text = ROOT.TLatex()
extra_text.SetNDC()
extra_text.SetTextFont(52)
extra_text.SetTextSize(0.04)
extra_text.DrawLatex(0.23, 0.91, "Preliminary Prompt Era C")

lumi_text = ROOT.TLatex()
lumi_text.SetNDC()
lumi_text.SetTextFont(42)
lumi_text.SetTextSize(0.04)
lumi_text.SetTextAlign(31)
lumi_text.DrawLatex(0.9, 0.91, "5.01 fb^{-1} (13.6 TeV)")

c1.cd()
pad2 = ROOT.TPad("pad2", "pad2", 0, 0.05, 1, 0.3)
pad2.SetTopMargin(0.05)
pad2.SetBottomMargin(0.3)
pad2.SetLeftMargin(0.15)
pad2.SetGridy()
pad2.Draw()
pad2.cd()

hist_ratio = hist_data.Clone("hist_ratio")
hist_ratio.Divide(hist_mc)

hist_ratio.SetMarkerStyle(ROOT.kFullCircle)
hist_ratio.SetMarkerSize(1.0)
hist_ratio.SetLineColor(ROOT.kBlack)
hist_ratio.SetTitle("")
hist_ratio.SetStats(0)

hist_ratio.GetYaxis().SetTitle("Data / MC")
hist_ratio.GetYaxis().SetNdivisions(505)
#hist_ratio.GetYaxis().SetRangeUser(0.5, 1.5) 
hist_ratio.GetYaxis().SetTitleSize(0.12)
hist_ratio.GetYaxis().SetTitleOffset(0.5)
hist_ratio.GetYaxis().SetLabelSize(0.1)
hist_ratio.GetYaxis().CenterTitle()

hist_ratio.GetXaxis().SetTitle("Number of True Interactions")
hist_ratio.GetXaxis().SetTitleSize(0.12)
hist_ratio.GetXaxis().SetTitleOffset(1.0)
hist_ratio.GetXaxis().SetLabelSize(0.1)
hist_ratio.Draw("E1")

line = ROOT.TLine(hist_ratio.GetXaxis().GetXmin(), 1, hist_ratio.GetXaxis().GetXmax(), 1)
line.SetLineColor(ROOT.kGray+2)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

c1.Update()
c1.Draw()
c1.SaveAs("pileup_comparison_ratio.png")
print("Gráfico 'pileup_comparison_ratio.png' salvo.")

Gráfico 'pileup_comparison_ratio.png' salvo.


Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: png file pileup_comparison_ratio.png has been created


In [8]:
nbins = hist_mc.GetNbinsX()
bins = np.empty([nbins])
mc_weight = np.empty([nbins])

for i in range(0, nbins):
    weight = hist_ratio.GetBinContent(i)
    bins[i] = i
    mc_weight[i] = weight

In [9]:
mc_weight

array([0.00000000e+00, 0.00000000e+00, 0.00000000e+00, 0.00000000e+00,
       1.10577743e+02, 1.33802458e+02, 2.60735564e+01, 1.79879542e+01,
       1.03056888e+01, 4.13061654e+00, 1.68426326e+00, 1.41089835e+00,
       7.83477511e-01, 5.33196609e-01, 3.71124693e-01, 2.33685414e-01,
       1.69085221e-01, 1.43873356e-01, 1.56987279e-01, 1.95456603e-01,
       2.90424414e-01, 4.42767517e-01, 5.91450184e-01, 7.17175151e-01,
       8.27782703e-01, 9.95797909e-01, 1.12112759e+00, 1.27649926e+00,
       1.40530251e+00, 1.51413013e+00, 1.60256632e+00, 1.70136428e+00,
       1.72775517e+00, 1.80809174e+00, 1.75077501e+00, 1.67755735e+00,
       1.56368057e+00, 1.44335933e+00, 1.30215660e+00, 1.15407132e+00,
       1.02536884e+00, 9.07034084e-01, 8.01854881e-01, 7.28483435e-01,
       6.52002977e-01, 5.85690758e-01, 5.30733642e-01, 4.76789932e-01,
       4.38889700e-01, 3.96583807e-01, 3.53241949e-01, 3.13440972e-01,
       2.79114608e-01, 2.50237232e-01, 2.16573307e-01, 1.94308894e-01,
      

In [10]:
np.save('mc_weight.npy', mc_weight)

In [11]:
ROOT.gInterpreter.Declare(f"const std::vector<double> pileup_weights = {{{','.join(map(str, mc_weight))}}};")
ROOT.gInterpreter.Declare("""
double getPileupWeight(float nTrueInt) {
    // Converte o número de interações em um índice de bin.
    int bin = static_cast<int>(nTrueInt + 0.5);

    // Garante que o índice está dentro dos limites do vetor de pesos
    if (bin >= 0 && bin < pileup_weights.size()) {
        return pileup_weights[bin];
    }
    // Se estiver fora do alcance, não aplica peso (peso = 1.0)
    return 1.0;
}
""")

True

In [16]:
df_mc_initial = ROOT.RDataFrame(chain)
df_mc_weighted = df_mc_initial.Define("pileupWeight", "getPileupWeight(Pileup_nTrueInt[0])")

nbins_data = hist_data.GetNbinsX()
xmin_data = hist_data.GetXaxis().GetXmin()
xmax_data = hist_data.GetXaxis().GetXmax()

nbins_data2 = hist_mc.GetNbinsX()
xmin_data2 = hist_mc.GetXaxis().GetXmin()
xmax_data2 = hist_mc.GetXaxis().GetXmax()

print(f"Histograma de dados detectado com {nbins_data} bins, no range [{xmin_data}, {xmax_data}]")
print(f"Histograma de MC detectado com {nbins_data2} bins, no range [{xmin_data2}, {xmax_data2}]")

mc_pu_model = ROOT.RDF.TH1DModel("mc_pu_model", "Pileup MC", nbins_data, xmin_data, xmax_data)
hist_mc_uncorrected = df_mc_initial.Histo1D(mc_pu_model, "Pileup_nTrueInt").GetValue()
df_mc_weighted = df_mc_initial.Define("pileupWeight", "getPileupWeight(Pileup_nTrueInt[0])")
hist_mc_corrected = df_mc_weighted.Histo1D(mc_pu_model, "Pileup_nTrueInt", "pileupWeight").GetValue()

hist_data.Scale(1.0 / hist_data.Integral())
hist_mc_uncorrected.Scale(1.0 / hist_mc_uncorrected.Integral())
hist_mc_corrected.Scale(1.0 / hist_mc_corrected.Integral())

hist_mc_corrected.SetLineColor(ROOT.kBlue)
hist_mc_corrected.SetLineWidth(2)
hist_mc_corrected.SetFillColorAlpha(ROOT.kBlue, 0.8)
hist_mc_corrected.SetStats(0)
hist_mc_corrected.SetTitle("")

hist_mc_uncorrected.SetLineColor(ROOT.kRed)
hist_mc_uncorrected.SetFillColorAlpha(ROOT.kRed, 0.35)
hist_mc_uncorrected.SetLineWidth(2)
hist_mc_uncorrected.SetStats(0)

hist_data.SetMarkerStyle(ROOT.kFullCircle)
hist_data.SetMarkerSize(1.0)
hist_data.SetLineColor(ROOT.kBlack)

c2 = ROOT.TCanvas("c2", "Pileup Closure Test", 800, 800)
pad3 = ROOT.TPad("pad3", "pad3", 0, 0.3, 1, 1.0)
pad3.SetBottomMargin(0.02)
pad3.SetLeftMargin(0.15)
pad3.Draw()

pad3.cd()
hist_mc_corrected.GetYaxis().SetTitle("Normalized Events")
hist_mc_corrected.GetYaxis().SetTitleSize(0.04)
hist_mc_corrected.GetYaxis().SetTitleOffset(1.3)
hist_mc_corrected.GetYaxis().SetLabelSize(0.04)
hist_mc_corrected.GetXaxis().SetLabelSize(0)
hist_mc_corrected.GetYaxis().SetRangeUser(0, hist_data.GetMaximum() * 1.5)
hist_data.SetStats(0)

hist_mc_corrected.Draw("HIST")
hist_mc_uncorrected.Draw("HIST SAME")
hist_data.Draw("E1 SAME")

legend = ROOT.TLegend(0.63, 0.76, 0.78, 0.88)
legend.AddEntry(hist_data, "Data", "lep")
legend.AddEntry(hist_mc_corrected, "MC Sim. (Corrigido)", "f")
legend.AddEntry(hist_mc_uncorrected, "MC Sim. (Original)", "f")
legend.SetBorderSize(0)
legend.SetTextFont(42)
legend.SetTextSize(0.04)
legend.Draw()

cms_text = ROOT.TLatex()
cms_text.SetNDC()
cms_text.SetTextFont(61)
cms_text.SetTextSize(0.05)
cms_text.DrawLatex(0.15, 0.91, "CMS")

extra_text = ROOT.TLatex()
extra_text.SetNDC()
extra_text.SetTextFont(52)
extra_text.SetTextSize(0.04)
extra_text.DrawLatex(0.23, 0.91, "Preliminary Prompt Era C")

lumi_text = ROOT.TLatex()
lumi_text.SetNDC()
lumi_text.SetTextFont(42)
lumi_text.SetTextSize(0.04)
lumi_text.SetTextAlign(31)
lumi_text.DrawLatex(0.9, 0.91, "5.01 fb^{-1} (13.6 TeV)")

c2.cd(); 
pad4 = ROOT.TPad("pad4", "pad4", 0, 0.05, 1, 0.3) #############################
pad4.SetTopMargin(0.05)
pad4.SetBottomMargin(0.3)
pad4.SetLeftMargin(0.15)
pad4.SetGridy()
pad4.Draw()

pad4.cd()
hist_ratio_corrected = hist_data.Clone("hist_ratio_corrected")
hist_ratio_corrected.Divide(hist_mc_corrected)
hist_ratio_corrected.SetMarkerStyle(ROOT.kFullCircle)
hist_ratio_corrected.SetMarkerSize(1.0)
hist_ratio_corrected.SetLineColor(ROOT.kBlack)
hist_ratio_corrected.SetTitle("")
hist_ratio_corrected.SetStats(0)
hist_ratio_corrected.GetYaxis().SetTitle("Data / MC_{Corrigido}")
hist_ratio_corrected.GetYaxis().SetNdivisions(505)
#hist_ratio_corrected.GetYaxis().SetRangeUser()
hist_ratio_corrected.GetYaxis().SetTitleSize(0.1)
hist_ratio_corrected.GetYaxis().SetTitleOffset(0.5)
hist_ratio_corrected.GetYaxis().SetLabelSize(0.1)
hist_ratio_corrected.GetYaxis().CenterTitle()
hist_ratio_corrected.GetXaxis().SetTitle("Number of True Interactions")
hist_ratio_corrected.GetXaxis().SetTitleSize(0.1)
hist_ratio_corrected.GetXaxis().SetTitleOffset(1.0)
hist_ratio_corrected.GetXaxis().SetLabelSize(0.1)
hist_ratio_corrected.Draw("E1")

line = ROOT.TLine(hist_ratio_corrected.GetXaxis().GetXmin(), 1, hist_ratio_corrected.GetXaxis().GetXmax(), 1)
line.SetLineColor(ROOT.kGray+2)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

output_filename = "pileup_comparison_corrected.png"
c2.SaveAs(output_filename)
c2.Draw()

Histograma de dados detectado com 100 bins, no range [0.0, 100.0]
Histograma de MC detectado com 100 bins, no range [0.0, 100.0]


Info in <TCanvas::Print>: png file pileup_comparison_corrected.png has been created


In [13]:
df_mc_initial.GetColumnType("Pileup_nTrueInt")

'ROOT::VecOps::RVec<Float_t>'

## Pileup usando Poisson

In [ ]:
file_data = ROOT.TFile("Pileup_data/MyDataPileupHistogram.root")
file_mc_cmssw = ROOT.TFile("Pileup_data/pileup_histogram.root")

hist_data = file_data.Get("pileup")
hist_mc_cmssw = file_mc_cmssw.Get("pileup_hist")

In [ ]:
if hist_data.GetNbinsX() != hist_mc_cmssw.GetNbinsX():
    if hist_data.GetNbinsX() > hist_mc_cmssw.GetNbinsX():
        hist_data.Rebin(hist_data.GetNbinsX() // hist_mc_cmssw.GetNbinsX())
    else:
        hist_mc_cmssw.Rebin(hist_mc_cmssw.GetNbinsX() // hist_data.GetNbinsX())

print(f'Número de bins no histograma de dados: {hist_data.GetNbinsX()}')
print(f'Número de bins no histograma de MC: {hist_mc_cmssw.GetNbinsX()}')


hist_data.Scale(1.0 / hist_data.Integral())
hist_mc_cmssw.Scale(1.0 / hist_mc_cmssw.Integral())

print(f'Área no histograma de dados: {hist_data.Integral()}')
print(f'Área no histograma de MC: {hist_mc_cmssw.Integral()}')

In [ ]:
c1 = ROOT.TCanvas("c1", "Pileup Comparison", 800, 800)
pad1 = ROOT.TPad("pad1", "pad1", 0, 0.3, 1, 1.0)
pad1.SetBottomMargin(0.02)
pad1.SetLeftMargin(0.15)
pad1.Draw()
pad1.cd()

hist_mc_cmssw.SetLineColor(ROOT.kRed)
hist_mc_cmssw.SetLineWidth(2)
hist_mc_cmssw.SetFillColorAlpha(ROOT.kRed, 0.35)
hist_mc_cmssw.SetTitle("")
hist_mc_cmssw.SetStats(0)

hist_data.SetMarkerStyle(ROOT.kFullCircle)
hist_data.SetMarkerSize(1.0)
hist_data.SetLineColor(ROOT.kBlack)

hist_mc_cmssw.GetYaxis().SetTitle("Normalized Events")
hist_mc_cmssw.GetYaxis().SetTitleSize(0.05)
hist_mc_cmssw.GetYaxis().SetTitleOffset(1.3)
hist_mc_cmssw.GetYaxis().SetLabelSize(0.04)
hist_mc_cmssw.GetXaxis().SetLabelSize(0)

hist_mc_cmssw.GetYaxis().SetRangeUser(0, 0.08)
hist_mc_cmssw.SetStats(0)
hist_data.SetStats(0)
hist_mc_cmssw.Draw("HIST")
hist_data.Draw("E1 SAME")

legend = ROOT.TLegend(0.65, 0.70, 0.88, 0.88)
legend.AddEntry(hist_data, "Data", "lep")
legend.AddEntry(hist_mc_cmssw, "MC Simulation", "f")
legend.SetBorderSize(0)
legend.SetTextFont(42)
legend.SetTextSize(0.04)
legend.Draw()

cms_text = ROOT.TLatex()
cms_text.SetNDC()
cms_text.SetTextFont(61)
cms_text.SetTextSize(0.05)
cms_text.DrawLatex(0.15, 0.91, "CMS")

extra_text = ROOT.TLatex()
extra_text.SetNDC()
extra_text.SetTextFont(52)
extra_text.SetTextSize(0.04)
extra_text.DrawLatex(0.23, 0.91, "Preliminary Prompt Era C")

lumi_text = ROOT.TLatex()
lumi_text.SetNDC()
lumi_text.SetTextFont(42)
lumi_text.SetTextSize(0.04)
lumi_text.SetTextAlign(31)
lumi_text.DrawLatex(0.9, 0.91, "5.01 fb^{-1} (13.6 TeV)")

c1.cd()
pad2 = ROOT.TPad("pad2", "pad2", 0, 0.05, 1, 0.3)
pad2.SetTopMargin(0.05)
pad2.SetBottomMargin(0.3)
pad2.SetLeftMargin(0.15)
pad2.SetGridy()
pad2.Draw()
pad2.cd()

hist_ratio = hist_data.Clone("hist_ratio")
hist_ratio.Divide(hist_mc_cmssw)

hist_ratio.SetMarkerStyle(ROOT.kFullCircle)
hist_ratio.SetMarkerSize(1.0)
hist_ratio.SetLineColor(ROOT.kBlack)
hist_ratio.SetTitle("")
hist_ratio.SetStats(0)

hist_ratio.GetYaxis().SetTitle("Data / MC")
hist_ratio.GetYaxis().SetNdivisions(505)
#hist_ratio.GetYaxis().SetRangeUser(0.5, 1.5) # Ajuste o range se necessário
hist_ratio.GetYaxis().SetTitleSize(0.12)
hist_ratio.GetYaxis().SetTitleOffset(0.5)
hist_ratio.GetYaxis().SetLabelSize(0.1)
hist_ratio.GetYaxis().CenterTitle()

hist_ratio.GetXaxis().SetTitle("Number of True Interactions")
hist_ratio.GetXaxis().SetTitleSize(0.12)
hist_ratio.GetXaxis().SetTitleOffset(1.0)
hist_ratio.GetXaxis().SetLabelSize(0.1)
hist_ratio.Draw("E1")

line = ROOT.TLine(hist_ratio.GetXaxis().GetXmin(), 1, hist_ratio.GetXaxis().GetXmax(), 1)
line.SetLineColor(ROOT.kGray+2)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

c1.Update()
c1.Draw()
c1.SaveAs("pileup_comparison_ratio_cmssw.png")
print("Gráfico 'pileup_comparison_ratio_cmssw.png' salvo.")

In [ ]:
nbins = hist_mc_cmssw.GetNbinsX()
bins = np.empty([nbins])
mc_weight_cmssw = np.empty([nbins])

for i in range(0, nbins):
    weight = hist_ratio.GetBinContent(i)
    bins[i] = i
    mc_weight_cmssw[i] = weight

In [ ]:
mc_weight_cmssw

In [ ]:
np.save('mc_weight_cmssw.npy', mc_weight_cmssw)

In [ ]:
pesos_pileup = np.load('mc_weight_cmssw.npy')

In [ ]:
h_dados = file_data.Get("pileup").Clone("h_dados_clone")
h_mc_original = file_mc_cmssw.Get("pileup_hist").Clone("h_mc_clone")
h_mc_corrigido = h_mc_original.Clone("h_mc_corrigido")

In [ ]:
for i in range(1, h_mc_corrigido.GetNbinsX() + 1):
    conteudo_original = h_mc_original.GetBinContent(i)
    erro_original = h_mc_original.GetBinError(i)
    peso = pesos_pileup[i-1]
    
    h_mc_corrigido.SetBinContent(i, conteudo_original * peso)
    h_mc_corrigido.SetBinError(i, erro_original * peso)

In [ ]:
h_dados.Scale(1.0 / h_dados.Integral())
h_mc_original.Scale(1.0 / h_mc_original.Integral())
h_mc_corrigido.Scale(1.0 / h_mc_corrigido.Integral())

h_mc_corrigido.SetLineColor(ROOT.kBlue)
h_mc_corrigido.SetLineWidth(2)
h_mc_corrigido.SetFillColorAlpha(ROOT.kBlue, 0.8)
h_mc_corrigido.SetStats(0)
h_mc_corrigido.SetTitle("")
h_mc_corrigido.SetLineColor(ROOT.kBlue)
h_mc_corrigido.SetLineWidth(2)
h_mc_corrigido.SetLineStyle(ROOT.kDashed)
h_mc_corrigido.SetStats(0)

h_dados.SetMarkerStyle(ROOT.kFullCircle)
h_dados.SetMarkerSize(1.0)
h_dados.SetLineColor(ROOT.kBlack)

c2 = ROOT.TCanvas("c2", "Pileup Closure Test", 800, 800)
pad3 = ROOT.TPad("pad3", "pad3", 0, 0.3, 1, 1.0)
pad3.SetBottomMargin(0.02)
pad3.SetLeftMargin(0.15)
pad3.Draw()

pad3.cd()
h_mc_corrigido.GetYaxis().SetTitle("Normalized Events")
h_mc_corrigido.GetYaxis().SetTitleSize(0.04)
h_mc_corrigido.GetYaxis().SetTitleOffset(1.3)
h_mc_corrigido.GetYaxis().SetLabelSize(0.04)
h_mc_corrigido.GetXaxis().SetLabelSize(0)
h_mc_corrigido.GetYaxis().SetRangeUser(0, hist_data.GetMaximum() * 1.5)
h_dados.SetStats(0)

h_mc_corrigido.Draw("HIST")
h_mc_original.Draw("HIST SAME")
h_dados.Draw("E1 SAME")

legend = ROOT.TLegend(0.63, 0.76, 0.78, 0.88)
legend.AddEntry(h_dados, "Data", "lep")
legend.AddEntry(h_mc_corrigido, "MC Sim. (Corrected)", "f")
legend.AddEntry(h_mc_original, "MC Sim. (Original)", "f")
legend.SetBorderSize(0)
legend.SetTextFont(42)
legend.SetTextSize(0.04)
legend.Draw()

cms_text = ROOT.TLatex()
cms_text.SetNDC()
cms_text.SetTextFont(61)
cms_text.SetTextSize(0.05)
cms_text.DrawLatex(0.15, 0.91, "CMS")

extra_text = ROOT.TLatex()
extra_text.SetNDC()
extra_text.SetTextFont(52)
extra_text.SetTextSize(0.04)
extra_text.DrawLatex(0.23, 0.91, "Preliminary Prompt Era C")

lumi_text = ROOT.TLatex()
lumi_text.SetNDC()
lumi_text.SetTextFont(42)
lumi_text.SetTextSize(0.04)
lumi_text.SetTextAlign(31)
lumi_text.DrawLatex(0.9, 0.91, "5.01 fb^{-1} (13.6 TeV)")

c2.cd(); 
pad4 = ROOT.TPad("pad4", "pad4", 0, 0.05, 1, 0.3) 
pad4.SetTopMargin(0.05)
pad4.SetBottomMargin(0.3)
pad4.SetLeftMargin(0.15)
pad4.SetGridy()
pad4.Draw()

pad4.cd()
hist_ratio_corrected = h_dados.Clone("hist_ratio_corrected")
hist_ratio_corrected.Divide(h_mc_corrigido)
hist_ratio_corrected.SetMarkerStyle(ROOT.kFullCircle)
hist_ratio_corrected.SetMarkerSize(1.0)
hist_ratio_corrected.SetLineColor(ROOT.kBlack)
hist_ratio_corrected.SetTitle("")
hist_ratio_corrected.SetStats(0)
hist_ratio_corrected.GetYaxis().SetTitle("Data / MC_{Corrected}")
hist_ratio_corrected.GetYaxis().SetNdivisions(505)
hist_ratio_corrected.GetYaxis().SetRangeUser(0, 2)
hist_ratio_corrected.GetYaxis().SetTitleSize(0.1)
hist_ratio_corrected.GetYaxis().SetTitleOffset(0.5)
hist_ratio_corrected.GetYaxis().SetLabelSize(0.1)
hist_ratio_corrected.GetYaxis().CenterTitle()
hist_ratio_corrected.GetXaxis().SetTitle("Number of True Interactions")
hist_ratio_corrected.GetXaxis().SetTitleSize(0.1)
hist_ratio_corrected.GetXaxis().SetTitleOffset(1.0)
hist_ratio_corrected.GetXaxis().SetLabelSize(0.1)
hist_ratio_corrected.Draw("E1")

line = ROOT.TLine(h_mc_corrigido.GetXaxis().GetXmin(), 1, h_mc_corrigido.GetXaxis().GetXmax(), 1)
line.SetLineColor(ROOT.kGray+2)
line.SetLineStyle(2)
line.SetLineWidth(2)
line.Draw("SAME")

output_filename = "pileup_comparison_corrected_cmssw.png"
c2.SaveAs(output_filename)
c2.Draw()
legend.SetBorderSize(0)